# Random Forest Regressor Model

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt # type: ignore

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier # type: ignore
from sklearn.datasets import make_moons # type: ignore
from sklearn.model_selection import train_test_split # type: ignore

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

ruta = r'C:\Users\jthow\iCloudDrive\Documents\3_Maestria_Estadistica_UNINORTE\3_Tercer_Semestre\Machine_Learning\tornados.csv.zip'  
df = pd.read_csv(ruta) 
df['loss'] = df['loss'].replace(0, pd.NA)
df['loss'] = df['loss'].interpolate(method='linear')
df['mag'] = df['mag'].fillna(df['mag'].mean())
df.isnull().sum()

om              0
yr              0
mo              0
dy              0
date            0
time            0
tz              0
datetime_utc    0
st              0
stf             0
mag             0
inj             0
fat             0
loss            0
slat            0
slon            0
elat            0
elon            0
len             0
wid             0
ns              0
sn              0
f1              0
f2              0
f3              0
f4              0
fc              0
dtype: int64

In [7]:
from random import Random
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Modelo de Random Forest Regressor
# Definir X y y (asegúrate de que ya tienes estas variables previamente definidas)
X = df[['mag', 'slat', 'slon', 'elat', 'elon', 'len', 'wid','f1', 'f2', 'f3', 'f4','loss']]
y = df['inj']

# Dividir los datos en conjunto de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

random = RandomForestRegressor().fit(X_train, y_train)
print("Training set score: {:.2f}".format(random.score(X_train, y_train)))
print("Test set score: {:.2f}".format(random.score(X_test, y_test)))
# Ver los coeficientes de cada variable en el modelo
# Obtener la importancia de las características
coeficientes = pd.Series(random.feature_importances_, index=X_train.columns)

# Mostrar la importancia de las características
print("Importancia de las características del modelo Random Forest:")
print(coeficientes)

Training set score: 0.90
Test set score: 0.42
Importancia de las características del modelo Random Forest:
mag     0.206482
slat    0.062337
slon    0.097412
elat    0.035964
elon    0.070258
len     0.080118
wid     0.077577
f1      0.063330
f2      0.028382
f3      0.015263
f4      0.007170
loss    0.255708
dtype: float64


## Metricas Random Forest Regressor Model

In [10]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
from joblib import dump, load
from time import time  # Agregar importación de time
from sklearn.ensemble import RandomForestRegressor  # Importar RandomForestRegressor
from sklearn.model_selection import GridSearchCV  # Importar GridSearchCV

# ------------------------
# Paso 2: Verificar si el modelo ya está guardado
# ------------------------
# Asegurarse de que el modelo esté cargado
try:
    grid_rf = load('grid_rf.joblib')  # Intentar cargar el modelo entrenado de Random Forest
    print("Modelo Random Forest cargado correctamente.")
except FileNotFoundError:
    print("No se encontró el modelo guardado. Entrenando el modelo desde cero...")
    
    # Si no se encuentra el modelo, entrenarlo desde cero
    # Aquí debes definir X_train, y_train antes de este paso

    rf_model = RandomForestRegressor()
    param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [10, 20, None]}  # Parametros para búsqueda
    grid_rf = GridSearchCV(rf_model, param_grid, cv=5)
    
    # Entrenar el modelo
    start_time = time()
    grid_rf.fit(X_train, y_train)
    training_time_rf = time() - start_time
    
    # Guardar el modelo entrenado
    dump(grid_rf, 'grid_rf.joblib')
    print("Modelo entrenado y guardado como 'grid_rf.joblib'.")

# ------------------------
# Paso 3: Medir el tiempo de entrenamiento
# ------------------------
if 'training_time_rf' not in globals():
    start_time = time()
    grid_rf.fit(X_train, y_train)
    training_time_rf = time() - start_time
else:
    training_time_rf = 0

# ------------------------
# Paso 4: Hacer predicciones con el modelo Random Forest
# ------------------------
# Predicciones con el mejor modelo de Random Forest
y_pred_rf = grid_rf.best_estimator_.predict(X_test)

# ------------------------
# Paso 5: Calcular las métricas para Random Forest
# ------------------------
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

# Ljung-Box p-value para Random Forest
lb_test_rf = acorr_ljungbox(y_test - y_pred_rf, lags=[10])
lb_p_value_rf = lb_test_rf['lb_pvalue'].iloc[0]  # Accede al valor p del primer lag (lag 10)

# Jarque-Bera p-value para Random Forest
jb_p_value_rf = jarque_bera(y_test - y_pred_rf)[1]

# ------------------------
# Paso 6: Crear DataFrames con los resultados para Random Forest
# ------------------------
# Resultados para Random Forest
resultados_rf = pd.DataFrame({
    'Modelo': ['Random Forest'],
    'MAPE': [f"{mape_rf:.2f}"],
    'RMSE': [f"{rmse_rf:.2f}"],
    'R Cuadrado': [f"{r2_rf:.2f}"],
    'Ljung-Box Test p-value': [f"{lb_p_value_rf:.4f}"],
    'Jarque-Bera p-value': [f"{jb_p_value_rf:.4f}"],
    'CPU time (s)': [round(training_time_rf, 2)]  # Tiempo de entrenamiento
})

# ------------------------
# Paso 7: Mostrar los resultados
# ------------------------
print("Métricas Random Forest:")
display(resultados_rf)


No se encontró el modelo guardado. Entrenando el modelo desde cero...
Modelo entrenado y guardado como 'grid_rf.joblib'.
Métricas Random Forest:


,Modelo,MAPE,RMSE,R Cuadrado,Ljung-Box Test p-value,Jarque-Bera p-value,CPU time (s)
0,Random Forest,1966536419626125.25,17.48,0.42,0.2712,0.0000,0
